In [53]:
import torch
from transformers import AutoTokenizer, set_seed
from parler_tts import ParlerTTSForConditionalGeneration, ParlerTTSConfig
import soundfile as sf
import os
import gc

# 1. Define Test Samples
# We pick two highly contrasting prompts to test the models' range
test_samples = [
    {
        "id": "test_1",
        "command": "Turn off the porch light please.",
        "description": "Thomas speaks sadly at a very slow speed with high quality audio."
    },
    {
        "id": "test_2",
        "command": "Turn off the porch light please.",
        "description": "Jerry speaks confusedly at a very fast speed with high quality audio."
    },
    {
        "id": "test_3",
        "command": "uh can you check if the kitchen bookshelf warm light is on or something",
        "description": "Elizabeth speaks happily at a slow speed with high quality audio."
    },
    {
        "id": "test_4",
        "command": "uh can you check if the kitchen bookshelf warm light is on or something",
        "description": "Jerry speaks neutrally at a very slow speed with high quality audio."
    },
]

In [54]:
gc.collect()
torch.cuda.empty_cache()

In [55]:
output_dir = "../test/model_comparison_tests"
os.makedirs(output_dir, exist_ok=True)
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 2. Reusable Generation Function
def run_model_test(model_id, short_name):
    print(f"\n--- Loading {model_id} ---")

    # Load model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    config = ParlerTTSConfig.from_pretrained(model_id)
    config.decoder._attn_implementation = "flash_attention_2"
    model = ParlerTTSForConditionalGeneration.from_pretrained(model_id, config=config).to(device)

    # Lock the seed so both models generate from the exact same random starting point
    set_seed(42)
    for sample in test_samples:
        print(f"Generating {sample['id']} with {short_name}...")

        # Tokenize inputs
        inputs = tokenizer(sample["description"], return_tensors="pt").to(device)
        prompt = tokenizer(sample["command"], return_tensors="pt").to(device)

        # Generate audio
        generation = model.generate(input_ids=inputs.input_ids,
                                    prompt_input_ids=prompt.input_ids,
                                    do_sample=True, # Enable sampling for more natural variance
                                    temperature=1.1 # Increase temperature for more creative generation
                                    )
        audio_arr = generation.cpu().numpy().squeeze()

        # Save file
        filename = f"{sample['id']}_{short_name}.wav"
        filepath = os.path.join(output_dir, filename)
        sf.write(filepath, audio_arr, model.config.sampling_rate)

    print(f"Finished {short_name}. Clearing VRAM...")

    # 3. Aggressive Memory Cleanup to prevent OOM errors
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()

Using device: cuda:0


In [56]:
# 3. Execute Tests Sequentially
# Test the Expresso-tuned mini model
run_model_test("parler-tts/parler-tts-mini-expresso", "MINI_EXPRESSO")


--- Loading parler-tts/parler-tts-mini-expresso ---


Config of the text_encoder: <class 'transformers.models.t5.modeling_t5.T5EncoderModel'> is overwritten by shared text_encoder config: T5Config {
  "_name_or_path": "google/flan-t5-base",
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2048,
  "d_kv": 64,
  "d_model": 768,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 12,
  "num_heads": 12,
  "num_layers": 12,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "task_specific_params": {
    "summarization": {
      "early_stopping": true,
      "length_penalty": 2.0,
      "max_length": 200,
      "min_length": 30,
      "no_repeat_ngram_si

Generating test_1 with MINI_EXPRESSO...
Generating test_2 with MINI_EXPRESSO...
Generating test_3 with MINI_EXPRESSO...
Generating test_4 with MINI_EXPRESSO...
Finished MINI_EXPRESSO. Clearing VRAM...
